# FeatureGraph evaluation: synthetic ground truth, baseline comparison, and sensitivity

This notebook evaluates FeatureGraph 0.2.0b1 as a behavioral-object
constructor. Unlike the demonstration notebook, it begins with signals
whose true troughs, peaks, boundaries, durations, amplitudes, and
symmetries are known.

The evaluation asks four questions:

1. Does FeatureGraph recover clean oscillation objects and their properties?
2. How does recovery change as observation noise increases?
3. How sensitive is recovery to smoothing, `diff_lag`, and `eps`?
4. How does object recovery compare with a SciPy peak/trough baseline?

All reported comparisons use deterministic seeds and one-to-one matching
between detected and true oscillations. FeatureGraph and the baseline are
evaluated against the same ground-truth objects.


In [ ]:
import hashlib
import json
import os
import platform
import subprocess
from dataclasses import dataclass
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.signal import find_peaks

import featuregraph as fg


def find_repository_root(start=None):
    current = Path.cwd() if start is None else Path(start)

    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise RuntimeError(
        "Could not locate the repository root containing pyproject.toml."
    )


REPOSITORY_ROOT = find_repository_root()
OUTPUT_DIR = REPOSITORY_ROOT / "artifacts" / "paper" / "evaluation"
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 1729
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

print(f"FeatureGraph {fg.__version__}; seed={SEED}")
print(f"Repository root: {REPOSITORY_ROOT}")


## 1. Evaluation protocol

A true oscillation is a complete trough–peak–trough interval. Detected
objects are matched to true objects by peak location, subject to a maximum
localization tolerance. Each detected and true object can be used at most
once.

Object recovery is summarized by precision, recall, and F1. For matched
objects, the notebook also reports mean absolute error (MAE) for start,
peak, end, duration, amplitude, and temporal symmetry.

The SciPy baseline detects positive and negative peaks and constructs an
object only when two consecutive detected troughs contain a detected peak.
This gives the baseline the same trough–peak–trough output schema required
for object-level comparison.


In [ ]:
@dataclass(frozen=True)
class SyntheticConfig:
    cycles: int = 12
    period: int = 80
    amplitude: float = 1.0
    baseline: float = 0.0
    noise_std: float = 0.0
    seed: int = SEED


def make_synthetic_oscillations(config):
    """Return observations and exact trough–peak–trough objects."""
    # A half-cycle of right-edge context makes the final true trough
    # observable for every evaluated difference lag. This context is
    # not included as an additional ground-truth object.
    index = np.arange(
        config.cycles * config.period + config.period // 2
    )
    clean = (
        config.baseline
        - config.amplitude
        * np.cos(2 * np.pi * index / config.period)
    )
    local_rng = np.random.default_rng(config.seed)
    observed = clean + local_rng.normal(
        0.0, config.noise_std, size=len(index)
    )

    observations = pd.DataFrame(
        {
            "sequence": 0,
            "sample_index": index,
            "signal_clean": clean,
            "signal": observed,
        }
    )

    starts = np.arange(config.cycles) * config.period
    peaks = starts + config.period // 2
    ends = starts + config.period
    truth = pd.DataFrame(
        {
            "true_id": np.arange(config.cycles),
            "start_index": starts,
            "peak_index": peaks,
            "end_index": ends,
            "duration": config.period,
            "amplitude": config.amplitude,
            "temporal_symmetry": 1.0,
        }
    )
    return observations, truth


observations, truth = make_synthetic_oscillations(SyntheticConfig())
truth.head()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(
    observations["sample_index"].iloc[:241],
    observations["signal"].iloc[:241],
    color="#3159a6",
)
ax.scatter(
    truth["peak_index"].iloc[:3],
    np.repeat(1.0, 3),
    color="#e76f51",
    label="True peaks",
    zorder=3,
)
ax.scatter(
    truth["start_index"].iloc[:4],
    np.repeat(-1.0, 4),
    color="#2a9d8f",
    label="True troughs",
    zorder=3,
)
ax.set(
    title="Synthetic ground truth: three complete oscillations",
    xlabel="Sample index",
    ylabel="Signal",
)
ax.legend()
fig.savefig(
    FIGURE_DIR / "synthetic_ground_truth.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


## 2. Construction and baseline adapters

These functions normalize both methods to the same object schema. The
FeatureGraph adapter returns only complete objects. The baseline uses
`find_peaks` for maxima and minima, then converts adjacent troughs with an
intervening peak into bounded oscillation objects.


In [ ]:
OBJECT_COLUMNS = [
    "start_index",
    "peak_index",
    "end_index",
    "duration",
    "amplitude",
    "temporal_symmetry",
]


def construct_featuregraph(
    observations,
    *,
    smooth_window=1,
    diff_lag=1,
    eps=0.0,
):
    constructor = fg.oscillation.Oscillation(
        signals="signal",
        group="sequence",
        smooth_signal=smooth_window > 1,
        smooth_window=smooth_window,
        diff_lag=diff_lag,
        eps=eps,
    )
    features = constructor.fit_transform(observations)
    transitions = fg.transition.Transition(
        signals="signal",
        group="sequence",
        diff_lag=diff_lag,
        eps=eps,
        source_signals={
            "signal": "signal_smooth",
        } if smooth_window > 1 else None,
    ).summarize(features, "signal")
    objects = constructor.summarize(features, "signal")

    assert transitions.table["is_complete"].all()
    assert set(transitions.table["direction"]) <= {
        "rising",
        "falling",
        "inactive",
    }
    return objects.table.loc[:, OBJECT_COLUMNS].copy(), features


def construct_scipy_baseline(
    observations,
    *,
    expected_period,
    distance_fraction=0.45,
    prominence=0.25,
):
    values = observations["signal"].to_numpy()
    distance = max(1, int(expected_period * distance_fraction))
    peak_indices, _ = find_peaks(
        values,
        distance=distance,
        prominence=prominence,
    )
    trough_indices, _ = find_peaks(
        -values,
        distance=distance,
        prominence=prominence,
    )
    if values[0] < values[1]:
        trough_indices = np.insert(trough_indices, 0, 0)

    rows = []
    for start, end in zip(trough_indices[:-1], trough_indices[1:]):
        candidates = peak_indices[
            (peak_indices > start) & (peak_indices < end)
        ]
        if len(candidates) == 0:
            continue
        peak = candidates[np.argmax(values[candidates])]
        rise = peak - start
        fall = end - peak
        duration = end - start
        rows.append(
            {
                "start_index": start,
                "peak_index": peak,
                "end_index": end,
                "duration": duration,
                "amplitude": (
                    values[peak]
                    - (values[start] + values[end]) / 2
                ) / 2,
                "temporal_symmetry": (
                    1 - abs(rise - fall) / duration
                ),
            }
        )
    return pd.DataFrame(rows, columns=OBJECT_COLUMNS)


## 3. One-to-one object matching and metrics

A detection is eligible for matching only when its peak is sufficiently
close to the true peak **and** its interval overlaps the true interval.
The primary protocol uses a peak tolerance of 10 samples and a minimum
interval intersection-over-union (IoU) of 0.50. Each detected and true
object can be used at most once.

Candidate pairs are ordered by peak error and interval overlap. Unmatched
detections are false positives; unmatched true objects are false negatives.
This prevents an object with a plausible peak but incorrect boundaries from
being counted as a correct recovery.


In [ ]:
PEAK_TOLERANCE = 10
MINIMUM_INTERVAL_IOU = 0.50


def interval_iou(detected_row, true_row):
    intersection_start = max(
        detected_row["start_index"],
        true_row["start_index"],
    )
    intersection_end = min(
        detected_row["end_index"],
        true_row["end_index"],
    )
    intersection = max(0, intersection_end - intersection_start)

    union_start = min(
        detected_row["start_index"],
        true_row["start_index"],
    )
    union_end = max(
        detected_row["end_index"],
        true_row["end_index"],
    )
    union = union_end - union_start

    return intersection / union if union > 0 else 0.0


def match_objects(
    detected,
    truth,
    *,
    peak_tolerance=PEAK_TOLERANCE,
    minimum_iou=MINIMUM_INTERVAL_IOU,
):
    candidates = []
    for detected_index, detected_row in detected.iterrows():
        for true_index, true_row in truth.iterrows():
            peak_error = abs(
                detected_row["peak_index"] - true_row["peak_index"]
            )
            overlap = interval_iou(detected_row, true_row)

            if (
                peak_error <= peak_tolerance
                and overlap >= minimum_iou
            ):
                candidates.append(
                    (
                        peak_error,
                        -overlap,
                        detected_index,
                        true_index,
                    )
                )

    used_detected = set()
    used_truth = set()
    matches = []
    for (
        peak_error,
        negative_overlap,
        detected_index,
        true_index,
    ) in sorted(candidates):
        if (
            detected_index in used_detected
            or true_index in used_truth
        ):
            continue

        used_detected.add(detected_index)
        used_truth.add(true_index)
        matches.append(
            {
                "detected_index": detected_index,
                "true_index": true_index,
                "peak_error": peak_error,
                "interval_iou": -negative_overlap,
            }
        )

    return matches


def score_objects(
    detected,
    truth,
    *,
    peak_tolerance=PEAK_TOLERANCE,
    minimum_iou=MINIMUM_INTERVAL_IOU,
):
    matches = match_objects(
        detected,
        truth,
        peak_tolerance=peak_tolerance,
        minimum_iou=minimum_iou,
    )
    true_positives = len(matches)
    false_positives = len(detected) - true_positives
    false_negatives = len(truth) - true_positives

    precision = (
        true_positives / (true_positives + false_positives)
        if true_positives + false_positives
        else 0.0
    )
    recall = (
        true_positives / (true_positives + false_negatives)
        if true_positives + false_negatives
        else 0.0
    )
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    result = {
        "detected": len(detected),
        "truth": len(truth),
        "matched": true_positives,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "interval_iou_mean": (
            float(np.mean([match["interval_iou"] for match in matches]))
            if matches
            else np.nan
        ),
    }

    error_columns = [
        "start_index",
        "peak_index",
        "end_index",
        "duration",
        "amplitude",
        "temporal_symmetry",
    ]
    for column in error_columns:
        errors = [
            abs(
                detected.loc[match["detected_index"], column]
                - truth.loc[match["true_index"], column]
            )
            for match in matches
        ]
        result[f"{column}_mae"] = (
            float(np.mean(errors)) if errors else np.nan
        )

    return result


def ci95(values):
    values = pd.Series(values).dropna()
    if len(values) < 2:
        return np.nan
    return 1.96 * values.std(ddof=1) / np.sqrt(len(values))


## 4. Clean-signal recovery

The clean-signal experiment is an implementation check. With `diff_lag=1`,
no smoothing, and zero tolerance, the generated extrema align exactly with
sample locations. A correct trough–peak–trough construction should recover
all complete objects without inventing extras.


In [ ]:
clean_detected, clean_features = construct_featuregraph(
    observations,
    smooth_window=1,
    diff_lag=1,
    eps=0.0,
)
clean_score = pd.Series(
    score_objects(
        clean_detected,
        truth,
    ),
    name="FeatureGraph",
)
clean_score


In [ ]:
clean_comparison = truth.merge(
    clean_detected,
    left_index=True,
    right_index=True,
    suffixes=("_true", "_detected"),
)
clean_comparison.head(8)


## 5. Parameter tuning on tuning seeds

FeatureGraph and SciPy are tuned using the same 20 synthetic signals at
noise standard deviation 0.20. The tuning seeds are never used in the final
robustness experiment.

FeatureGraph varies smoothing window, difference lag, and tolerance. SciPy
varies minimum peak distance and prominence. Both operating points are
selected by the same declared rule: highest mean object F1, followed by
lowest peak MAE and duration MAE.


In [ ]:
PERIOD = 80
TUNING_NOISE_STD = 0.20
TUNING_SEEDS = [SEED + replicate for replicate in range(20)]
TEST_SEEDS = [SEED + 10_000 + replicate for replicate in range(30)]

SMOOTH_WINDOWS = [1, 5, 9, 15]
DIFF_LAGS = [1, 3, 5, 10]
EPS_VALUES = [0.0, 0.01, 0.03]

SCIPY_DISTANCE_FRACTIONS = [0.30, 0.45, 0.60]
SCIPY_PROMINENCES = [0.10, 0.25, 0.40]

featuregraph_tuning_rows = []
for smooth_window, diff_lag, eps, experiment_seed in product(
    SMOOTH_WINDOWS,
    DIFF_LAGS,
    EPS_VALUES,
    TUNING_SEEDS,
):
    tuning_observations, tuning_truth = make_synthetic_oscillations(
        SyntheticConfig(
            period=PERIOD,
            noise_std=TUNING_NOISE_STD,
            seed=experiment_seed,
        )
    )
    detected, _ = construct_featuregraph(
        tuning_observations,
        smooth_window=smooth_window,
        diff_lag=diff_lag,
        eps=eps,
    )
    featuregraph_tuning_rows.append(
        {
            "smooth_window": smooth_window,
            "diff_lag": diff_lag,
            "eps": eps,
            "seed": experiment_seed,
            **score_objects(detected, tuning_truth),
        }
    )

featuregraph_tuning = pd.DataFrame(featuregraph_tuning_rows)
featuregraph_tuning_summary = (
    featuregraph_tuning.groupby(
        ["smooth_window", "diff_lag", "eps"],
        sort=False,
    )
    .agg(
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        f1_ci95=("f1", ci95),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        interval_iou_mean=("interval_iou_mean", "mean"),
        peak_mae=("peak_index_mae", "mean"),
        duration_mae=("duration_mae", "mean"),
        temporal_symmetry_mae=("temporal_symmetry_mae", "mean"),
    )
    .reset_index()
    .sort_values(
        [
            "f1_mean",
            "peak_mae",
            "duration_mae",
            "smooth_window",
            "diff_lag",
            "eps",
        ],
        ascending=[False, True, True, True, True, True],
    )
    .reset_index(drop=True)
)

selected_featuregraph = featuregraph_tuning_summary.iloc[0]
FEATUREGRAPH_OPERATING_POINT = {
    "smooth_window": int(selected_featuregraph["smooth_window"]),
    "diff_lag": int(selected_featuregraph["diff_lag"]),
    "eps": float(selected_featuregraph["eps"]),
}

scipy_tuning_rows = []
for distance_fraction, prominence, experiment_seed in product(
    SCIPY_DISTANCE_FRACTIONS,
    SCIPY_PROMINENCES,
    TUNING_SEEDS,
):
    tuning_observations, tuning_truth = make_synthetic_oscillations(
        SyntheticConfig(
            period=PERIOD,
            noise_std=TUNING_NOISE_STD,
            seed=experiment_seed,
        )
    )
    detected = construct_scipy_baseline(
        tuning_observations,
        expected_period=PERIOD,
        distance_fraction=distance_fraction,
        prominence=prominence,
    )
    scipy_tuning_rows.append(
        {
            "distance_fraction": distance_fraction,
            "prominence": prominence,
            "seed": experiment_seed,
            **score_objects(detected, tuning_truth),
        }
    )

scipy_tuning = pd.DataFrame(scipy_tuning_rows)
scipy_tuning_summary = (
    scipy_tuning.groupby(
        ["distance_fraction", "prominence"],
        sort=False,
    )
    .agg(
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        f1_ci95=("f1", ci95),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        interval_iou_mean=("interval_iou_mean", "mean"),
        peak_mae=("peak_index_mae", "mean"),
        duration_mae=("duration_mae", "mean"),
        temporal_symmetry_mae=("temporal_symmetry_mae", "mean"),
    )
    .reset_index()
    .sort_values(
        [
            "f1_mean",
            "peak_mae",
            "duration_mae",
            "distance_fraction",
            "prominence",
        ],
        ascending=[False, True, True, True, True],
    )
    .reset_index(drop=True)
)

selected_scipy = scipy_tuning_summary.iloc[0]
SCIPY_OPERATING_POINT = {
    "distance_fraction": float(
        selected_scipy["distance_fraction"]
    ),
    "prominence": float(selected_scipy["prominence"]),
}

pd.DataFrame(
    [
        {"method": "FeatureGraph", **FEATUREGRAPH_OPERATING_POINT},
        {"method": "SciPy baseline", **SCIPY_OPERATING_POINT},
    ]
)


In [ ]:
heatmap_table = (
    featuregraph_tuning.loc[featuregraph_tuning["eps"].eq(0.0)]
    .groupby(["smooth_window", "diff_lag"])["f1"]
    .mean()
    .unstack("diff_lag")
)

fig, ax = plt.subplots(figsize=(7, 4.5))
image = ax.imshow(
    heatmap_table.to_numpy(),
    vmin=0,
    vmax=1,
    cmap="viridis",
    aspect="auto",
)
ax.set_xticks(
    range(len(heatmap_table.columns)),
    labels=heatmap_table.columns,
)
ax.set_yticks(
    range(len(heatmap_table.index)),
    labels=heatmap_table.index,
)
ax.set(
    title=(
        "FeatureGraph tuning at noise std = 0.20 and eps = 0"
    ),
    xlabel="Difference lag",
    ylabel="Smoothing window",
)
for row in range(heatmap_table.shape[0]):
    for column in range(heatmap_table.shape[1]):
        value = heatmap_table.iloc[row, column]
        ax.text(
            column,
            row,
            f"{value:.2f}",
            ha="center",
            va="center",
            color="white" if value < 0.55 else "black",
        )
fig.colorbar(image, ax=ax, label="Mean object F1")
fig.savefig(
    FIGURE_DIR / "parameter_sensitivity.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


## 6. Held-out noise robustness

The selected operating points are frozen before this experiment. Performance
is evaluated on 30 new seeds at every noise level. None of these seeds were
used for parameter selection.

Error bars show 95% confidence intervals for mean object F1. The tables retain
both standard deviation and confidence intervals.


In [ ]:
NOISE_LEVELS = [0.00, 0.05, 0.10, 0.20, 0.30, 0.40]

robustness_rows = []
for noise_std, experiment_seed in product(
    NOISE_LEVELS,
    TEST_SEEDS,
):
    test_observations, test_truth = make_synthetic_oscillations(
        SyntheticConfig(
            period=PERIOD,
            noise_std=noise_std,
            seed=experiment_seed,
        )
    )

    featuregraph_detected, _ = construct_featuregraph(
        test_observations,
        **FEATUREGRAPH_OPERATING_POINT,
    )
    scipy_detected = construct_scipy_baseline(
        test_observations,
        expected_period=PERIOD,
        **SCIPY_OPERATING_POINT,
    )

    for method, detected in [
        ("FeatureGraph", featuregraph_detected),
        ("SciPy baseline", scipy_detected),
    ]:
        robustness_rows.append(
            {
                "noise_std": noise_std,
                "seed": experiment_seed,
                "method": method,
                **score_objects(detected, test_truth),
            }
        )

robustness = pd.DataFrame(robustness_rows)
robustness_summary = (
    robustness.groupby(["method", "noise_std"], sort=False)
    .agg(
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        f1_ci95=("f1", ci95),
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        interval_iou_mean=("interval_iou_mean", "mean"),
        peak_mae=("peak_index_mae", "mean"),
        start_mae=("start_index_mae", "mean"),
        end_mae=("end_index_mae", "mean"),
        duration_mae=("duration_mae", "mean"),
        amplitude_mae=("amplitude_mae", "mean"),
        temporal_symmetry_mae=(
            "temporal_symmetry_mae",
            "mean",
        ),
    )
    .reset_index()
)
robustness_summary


In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(12, 4), constrained_layout=True
)
for method, method_table in robustness_summary.groupby(
    "method", sort=False
):
    axes[0].errorbar(
        method_table["noise_std"],
        method_table["f1_mean"],
        yerr=method_table["f1_ci95"].fillna(0),
        marker="o",
        capsize=3,
        label=method,
    )
    axes[1].plot(
        method_table["noise_std"],
        method_table["interval_iou_mean"],
        marker="o",
        label=method,
    )

axes[0].set(
    title="Held-out object recovery under noise",
    xlabel="Noise standard deviation",
    ylabel="Object F1",
    ylim=(-0.02, 1.02),
)
axes[1].set(
    title="Boundary overlap under noise",
    xlabel="Noise standard deviation",
    ylabel="Mean interval IoU",
    ylim=(-0.02, 1.02),
)
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.25)
fig.savefig(
    FIGURE_DIR / "noise_robustness.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


## 7. Boundary completeness and group isolation

The final diagnostic checks two structural invariants:

- boundary-truncated intervals are excluded from the default object table;
- construction state does not cross independent group boundaries.

Two sequences are concatenated into one DataFrame. Each begins at a
different phase, so an implementation that leaks `diff` or cumulative
identity across the boundary can create a spurious object.


In [ ]:
group_frames = []
expected_by_group = {}
for group_id, phase_shift in [(0, 0), (1, 17)]:
    config = SyntheticConfig(cycles=6, period=PERIOD, seed=SEED)
    group_observations, _ = make_synthetic_oscillations(config)
    shifted_index = (
        group_observations["sample_index"] + phase_shift
    )
    group_observations["signal"] = -np.cos(
        2 * np.pi * shifted_index / PERIOD
    )
    group_observations["sequence"] = group_id
    group_frames.append(group_observations)

grouped_observations = pd.concat(
    group_frames, ignore_index=True
)
grouped_constructor = fg.oscillation.Oscillation(
    signals="signal",
    group="sequence",
    smooth_signal=False,
    diff_lag=1,
    eps=0.0,
)
grouped_features = grouped_constructor.fit_transform(
    grouped_observations
)
grouped_objects = grouped_constructor.summarize(
    grouped_features,
    "signal",
    include_partial=True,
)

structural_diagnostic = (
    grouped_objects.table.groupby("sequence")
    .agg(
        all_objects=("oscillation_id", "size"),
        complete_objects=("is_complete", "sum"),
        partial_objects=(
            "is_complete",
            lambda values: (~values).sum(),
        ),
    )
    .reset_index()
)
structural_diagnostic


In [ ]:
complete_grouped_objects = grouped_constructor.summarize(
    grouped_features,
    "signal",
    include_partial=False,
)

assert clean_score["precision"] == 1.0
assert clean_score["recall"] == 1.0
assert clean_score["f1"] == 1.0
assert clean_score["start_index_mae"] == 0.0
assert clean_score["peak_index_mae"] == 0.0
assert clean_score["end_index_mae"] == 0.0
assert clean_score["interval_iou_mean"] == 1.0
assert complete_grouped_objects.table["is_complete"].all()
assert set(complete_grouped_objects.table["sequence"]) == {0, 1}
assert set(TUNING_SEEDS).isdisjoint(TEST_SEEDS)
assert len(featuregraph_tuning) == (
    len(SMOOTH_WINDOWS)
    * len(DIFF_LAGS)
    * len(EPS_VALUES)
    * len(TUNING_SEEDS)
)
assert len(scipy_tuning) == (
    len(SCIPY_DISTANCE_FRACTIONS)
    * len(SCIPY_PROMINENCES)
    * len(TUNING_SEEDS)
)
assert len(robustness) == (
    len(NOISE_LEVELS)
    * len(TEST_SEEDS)
    * 2
)

print("All evaluation and structural assertions passed.")


## 8. Paper-ready outputs

The following cell writes tuning results, selected operating points, held-out
robustness results, structural diagnostics, figures, and a complete
reproducibility manifest. The manifest records package versions, independent
seed sets, matching rules, source revision, and SHA-256 hashes for every
generated artifact except the manifest itself.

These are derived outputs and must not be edited manually.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def git_value(*arguments):
    result = subprocess.run(
        ["git", *arguments],
        cwd=REPOSITORY_ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    return result.stdout.strip() or "not-available"


clean_score.to_frame("value").to_csv(
    OUTPUT_DIR / "clean_recovery.csv"
)
featuregraph_tuning.to_csv(
    OUTPUT_DIR / "featuregraph_tuning_replicates.csv",
    index=False,
)
featuregraph_tuning_summary.to_csv(
    OUTPUT_DIR / "featuregraph_tuning_summary.csv",
    index=False,
)
scipy_tuning.to_csv(
    OUTPUT_DIR / "scipy_tuning_replicates.csv",
    index=False,
)
scipy_tuning_summary.to_csv(
    OUTPUT_DIR / "scipy_tuning_summary.csv",
    index=False,
)
robustness.to_csv(
    OUTPUT_DIR / "noise_robustness_replicates.csv",
    index=False,
)
robustness_summary.to_csv(
    OUTPUT_DIR / "noise_robustness_summary.csv",
    index=False,
)
structural_diagnostic.to_csv(
    OUTPUT_DIR / "structural_diagnostic.csv",
    index=False,
)

selected_operating_points = {
    "FeatureGraph": FEATUREGRAPH_OPERATING_POINT,
    "SciPy baseline": SCIPY_OPERATING_POINT,
}
(OUTPUT_DIR / "selected_operating_points.json").write_text(
    json.dumps(selected_operating_points, indent=2, sort_keys=True)
    + "\n",
    encoding="utf-8",
)

manifest_path = OUTPUT_DIR / "evaluation_manifest.json"
artifact_paths = sorted(
    path
    for path in OUTPUT_DIR.rglob("*")
    if path.is_file() and path != manifest_path
)

source_commit = os.environ.get(
    "FEATUREGRAPH_SOURCE_COMMIT",
    git_value("rev-parse", "HEAD"),
)
manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "featuregraph_version": fg.__version__,
    "source_release": "v0.2.0b1",
    "source_commit": source_commit,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scipy_version": scipy.__version__,
    "matplotlib_version": matplotlib.__version__,
    "base_seed": SEED,
    "tuning_seeds": TUNING_SEEDS,
    "test_seeds": TEST_SEEDS,
    "cycles_per_signal": SyntheticConfig().cycles,
    "period_samples": PERIOD,
    "tuning_noise_std": TUNING_NOISE_STD,
    "test_noise_levels": NOISE_LEVELS,
    "matching_protocol": {
        "peak_tolerance_samples": PEAK_TOLERANCE,
        "minimum_interval_iou": MINIMUM_INTERVAL_IOU,
    },
    "featuregraph_tuning_grid": {
        "smooth_windows": SMOOTH_WINDOWS,
        "diff_lags": DIFF_LAGS,
        "eps_values": EPS_VALUES,
    },
    "scipy_tuning_grid": {
        "distance_fractions": SCIPY_DISTANCE_FRACTIONS,
        "prominences": SCIPY_PROMINENCES,
    },
    "selected_operating_points": selected_operating_points,
    "artifacts": {
        str(path.relative_to(REPOSITORY_ROOT)): sha256(path)
        for path in artifact_paths
    },
}
manifest_path.write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(f"Wrote evaluation artifacts to {OUTPUT_DIR.resolve()}")


## Interpretation and limitations

This notebook evaluates deterministic recovery of a declared synthetic
oscillation definition. It supports claims about boundary construction,
object properties, parameter sensitivity, and degradation under additive
Gaussian noise.

Parameter selection and final evaluation use disjoint seed sets. The reported
robustness results are therefore held out from tuning. Matching requires both
peak proximity and interval overlap rather than peak proximity alone.

The experiment does **not** establish that every real waveform has a unique
correct trough–peak–trough interpretation. It also does not evaluate irregular
timestamps, missing observations, nested frequencies, or expert agreement on
physiological and industrial data. Those require separate experiments.

The SciPy comparison is a detector baseline, not a comparison with another
behavioral-object framework. Its purpose is to separate extrema-detection
performance from FeatureGraph's additional representational output:
identities, completeness, properties, construction evidence, composition,
and queries.
